<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Computer%20Vision/Bevezet%C3%A9s%20a%20Self%20Supervised%20Learningbe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bevezetés a Self-Supervised Learning-be (Vision)

A **Self-Supervised Learning (SSL)** címkézetlen adatokból tanul reprezentációkat, mesterségesen generált felügyelettel. A computer vision területén forradalmi áttörést hozott.

## Tartalomjegyzék

1. SSL motiváció és alapelvek
2. Pretext feladatok
3. Kontraszt tanulás (SimCLR)
4. Modern SSL módszerek

# Bevezetés a Self-Supervised Learning-be (Vision)

A **Self-Supervised Learning (SSL)** egy tanulási paradigma, ahol a modell **címkézetlen adatokból** tanul hasznos reprezentációkat. A "felügyeletet" a rendszer maga generálja az adatok struktúrájából.

## Miért fontos?

- **Címkézés drága**: ImageNet 14M kép címkézése évekig tartott
- **Rengeteg címkézetlen adat**: Internet, videók, orvosi képek
- **Jobb reprezentációk**: SSL modellek gyakran jobban általánosítanak

## Tartalomjegyzék

1. SSL alapelvek és taxonómia
2. Pretext feladatok (korai módszerek)
3. Kontraszt tanulás (SimCLR, MoCo)
4. Modern módszerek (BYOL, DINO, MAE)
5. Gyakorlati használat

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
torch.manual_seed(42)
np.random.seed(42)

## 1. SSL alapelvek

### Tanulási paradigmák összehasonlítása

| Paradigma | Címke | Példa |
|-----------|-------|-------|
| Supervised | Emberi annotáció | "Ez egy kutya" |
| Unsupervised | Nincs | Klaszterezés |
| **Self-Supervised** | Automatikusan generált | "Ez a kép elforgatva van" |

### SSL két fő családja

```
Self-Supervised Learning
├── Pretext Tasks (2015-2018)
│   ├── Rotation prediction
│   ├── Jigsaw puzzle
│   └── Colorization
│
└── Contrastive Learning (2018-)
    ├── SimCLR, MoCo (negatív mintákkal)
    ├── BYOL, SimSiam (negatívok nélkül)
    └── DINO, MAE (self-distillation, masking)
```

In [ ]:
# Teszt képek létrehozása
def create_sample_images():
    images = []

    # 1. Geometriai alakzatok
    img = Image.new('RGB', (128, 128), (240, 245, 250))
    draw = ImageDraw.Draw(img)
    draw.ellipse([30, 30, 98, 98], fill=(200, 70, 70))
    images.append(('Kör', img))

    # 2. Komplex kép
    img = Image.new('RGB', (128, 128), (240, 245, 250))
    draw = ImageDraw.Draw(img)
    draw.rectangle([20, 20, 60, 60], fill=(70, 150, 70))
    draw.ellipse([60, 60, 110, 110], fill=(70, 70, 180))
    draw.polygon([(64, 10), (40, 50), (88, 50)], fill=(200, 180, 50))
    images.append(('Vegyes', img))

    return images

samples = create_sample_images()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, img) in zip(axes, samples):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Teszt képek az SSL demonstrációhoz', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Pretext feladatok

A **pretext task** egy mesterséges feladat, ahol a címkéket automatikusan generáljuk. A cél: a feladat megoldásához a modellnek meg kell tanulnia a kép strukturális jellemzőit.

### Rotation Prediction (Gidaris et al., 2018)

**Ötlet**: Forgassuk el a képet 0°, 90°, 180° vagy 270°-kal, és tanítsuk a modellt a forgatás felismerésére.

**Miért működik?** A forgatás felismeréséhez a modellnek "értenie" kell a képet (mi van fent/lent, objektumok orientációja).

In [ ]:
def create_rotation_task(img):
    """Rotation prediction pretext task."""
    angles = [0, 90, 180, 270]
    rotated_images = []
    labels = []

    for label, angle in enumerate(angles):
        rotated = img.rotate(angle)
        rotated_images.append(rotated)
        labels.append(label)

    return rotated_images, labels, angles

# Vizualizáció
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
rotated, labels, angles = create_rotation_task(samples[1][1])

for ax, img, label, angle in zip(axes, rotated, labels, angles):
    ax.imshow(img)
    ax.set_title(f'Címke: {label}\n({angle}° forgatás)', fontsize=10)
    ax.axis('off')

plt.suptitle('Rotation Prediction Pretext Task', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("A modell feladata: megtanulni, melyik kép mennyire van elforgatva.")
print("Ehhez 'értenie' kell a képet!")

In [ ]:
# Jigsaw Puzzle pretext task
def create_jigsaw_task(img, grid_size=3):
    """Jigsaw puzzle: kép feldarabolása és összekeverése."""
    img_array = np.array(img)
    h, w = img_array.shape[:2]
    ph, pw = h // grid_size, w // grid_size

    # Patch-ek kivágása
    patches = []
    for i in range(grid_size):
        for j in range(grid_size):
            patch = img_array[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
            patches.append(patch)

    # Összekeverés
    perm = np.random.permutation(len(patches))
    shuffled = [patches[i] for i in perm]

    # Visszarakás
    rows = [np.concatenate(shuffled[i*grid_size:(i+1)*grid_size], axis=1)
            for i in range(grid_size)]
    shuffled_img = np.concatenate(rows, axis=0)

    return Image.fromarray(shuffled_img), perm, patches

# Vizualizáció
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(samples[1][1])
axes[0].set_title('Eredeti kép', fontsize=11)
axes[0].axis('off')

for i in range(1, 4):
    shuffled, perm, _ = create_jigsaw_task(samples[1][1], 3)
    axes[i].imshow(shuffled)
    axes[i].set_title(f'Kevert #{i}\nperm: {list(perm)}', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Jigsaw Puzzle Pretext Task', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Kontraszt tanulás (Contrastive Learning)

### Alapötlet

- **Pozitív pár**: Ugyanannak a képnek két különböző augmentációja → közel a reprezentációs térben
- **Negatív pár**: Különböző képek → távol egymástól

### SimCLR (Chen et al., 2020)

$$\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$

ahol $\text{sim}(u, v) = \frac{u^T v}{\|u\| \|v\|}$ a cosine hasonlóság és $\tau$ a hőmérséklet.

In [ ]:
# SimCLR augmentáció pipeline
simclr_augment = T.Compose([
    T.RandomResizedCrop(128, scale=(0.4, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.RandomApply([T.GaussianBlur(kernel_size=9)], p=0.5),
])

# Pozitív párok vizualizálása
fig, axes = plt.subplots(3, 6, figsize=(14, 7))

for row in range(3):
    original = samples[1][1]

    # Eredeti (csak az első oszlopban)
    if row == 0:
        axes[row, 0].imshow(original)
        axes[row, 0].set_title('Eredeti', fontsize=10)
    else:
        axes[row, 0].axis('off')

    # 5 különböző augmentáció
    for col in range(1, 6):
        aug = simclr_augment(original)
        axes[row, col].imshow(aug)
        if row == 0:
            axes[row, col].set_title(f'Aug {col}', fontsize=10)
        axes[row, col].axis('off')

plt.suptitle('SimCLR augmentációk - Minden pár "pozitív" (ugyanaz a kép)',
            fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SimCLR encoder architektúra
class SimCLREncoder(nn.Module):
    """
    SimCLR = Backbone + Projection Head

    A projection head csak a tanítás során kell!
    Downstream feladatoknál a backbone kimenetét használjuk.
    """
    def __init__(self, backbone_dim=512, proj_dim=128):
        super().__init__()

        # Backbone (egyszerűsített CNN)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1),   # 128→64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1),  # 64→32
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, 2, 1), # 32→16
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, backbone_dim, 3, 2, 1), # 16→8
            nn.BatchNorm2d(backbone_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        # Projection head (MLP)
        self.projector = nn.Sequential(
            nn.Linear(backbone_dim, backbone_dim),
            nn.ReLU(),
            nn.Linear(backbone_dim, proj_dim)
        )

    def forward(self, x, return_projection=True):
        h = self.backbone(x)  # Reprezentáció
        if return_projection:
            z = self.projector(h)  # Projekció (tanításhoz)
            return F.normalize(z, dim=1)  # L2 normalizálás
        return h

# NT-Xent loss (InfoNCE)
def nt_xent_loss(z1, z2, temperature=0.5):
    """
    Normalized Temperature-scaled Cross Entropy Loss.
    z1, z2: Pozitív párok (batch_size × proj_dim)
    """
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # 2N × dim

    # Cosine hasonlóság mátrix
    sim = torch.mm(z, z.t()) / temperature  # 2N × 2N

    # Diagonális maszkolása (önmagával ne hasonlítson)
    mask = torch.eye(2 * batch_size, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)

    # Pozitív párok indexei: (i, i+N) és (i+N, i)
    labels = torch.cat([torch.arange(batch_size) + batch_size,
                       torch.arange(batch_size)]).long()

    return F.cross_entropy(sim, labels)

# Teszt
encoder = SimCLREncoder()
x = torch.randn(8, 3, 128, 128)
z = encoder(x)
print(f"Input: {x.shape} → Projection: {z.shape}")

# Loss teszt
z1 = torch.randn(4, 128)
z2 = torch.randn(4, 128)
loss = nt_xent_loss(F.normalize(z1), F.normalize(z2))
print(f"NT-Xent loss: {loss.item():.4f}")

### Kontraszt tanulás vizuálisan

Mit csinál a loss? A pozitív párokat "összehúzza", a negatívokat "eltolja".

In [ ]:
# Kontraszt tanulás vizualizáció
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

np.random.seed(42)

# 1. Kezdeti állapot (random)
n_images = 4
colors = plt.cm.Set1(np.linspace(0, 1, n_images))

embeddings = np.random.randn(n_images * 2, 2)  # 2 view per image

ax = axes[0]
for i in range(n_images):
    ax.scatter(embeddings[i, 0], embeddings[i, 1], c=[colors[i]], s=150, marker='o', label=f'Kép {i+1} - View 1')
    ax.scatter(embeddings[n_images+i, 0], embeddings[n_images+i, 1], c=[colors[i]], s=150, marker='^', label=f'Kép {i+1} - View 2')
ax.set_title('Kezdeti állapot\n(random embedding)', fontsize=11)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)

# 2. Tanítás közben
embeddings_mid = embeddings.copy()
for i in range(n_images):
    # Pozitív párok közelítése
    center = (embeddings_mid[i] + embeddings_mid[n_images+i]) / 2
    embeddings_mid[i] = embeddings_mid[i] * 0.7 + center * 0.3
    embeddings_mid[n_images+i] = embeddings_mid[n_images+i] * 0.7 + center * 0.3

ax = axes[1]
for i in range(n_images):
    ax.scatter(embeddings_mid[i, 0], embeddings_mid[i, 1], c=[colors[i]], s=150, marker='o')
    ax.scatter(embeddings_mid[n_images+i, 0], embeddings_mid[n_images+i, 1], c=[colors[i]], s=150, marker='^')
    # Nyíl a pozitív pár között
    ax.annotate('', xy=embeddings_mid[n_images+i], xytext=embeddings_mid[i],
               arrowprops=dict(arrowstyle='->', color=colors[i], lw=2, alpha=0.5))
ax.set_title('Tanítás közben\n(pozitív párok közelednek)', fontsize=11)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)

# 3. Tanítás után
theta = np.linspace(0, 2*np.pi, n_images+1)[:-1]
radius = 2
embeddings_final = np.zeros((n_images * 2, 2))
for i in range(n_images):
    center = np.array([radius * np.cos(theta[i]), radius * np.sin(theta[i])])
    embeddings_final[i] = center + np.random.randn(2) * 0.1
    embeddings_final[n_images+i] = center + np.random.randn(2) * 0.1

ax = axes[2]
for i in range(n_images):
    ax.scatter(embeddings_final[i, 0], embeddings_final[i, 1], c=[colors[i]], s=150, marker='o')
    ax.scatter(embeddings_final[n_images+i, 0], embeddings_final[n_images+i, 1], c=[colors[i]], s=150, marker='^')
ax.set_title('Tanítás után\n(klaszterek: pozitív párok együtt)', fontsize=11)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()

print("○ = View 1, △ = View 2")
print("Azonos szín = ugyanaz a kép (pozitív pár)")

## 4. Modern módszerek

### Fejlődési vonal

| Módszer | Év | Fő újítás | ImageNet Linear |
|---------|-----|-----------|----------------|
| Rotation | 2018 | Pretext task | ~55% |
| SimCLR | 2020 | Erős augment + nagy batch | 76.5% |
| MoCo v2 | 2020 | Momentum encoder + queue | 77.3% |
| BYOL | 2020 | Nincs negatív minta! | 79.6% |
| DINO | 2021 | Self-distillation + ViT | 83.6% |
| MAE | 2022 | Masked autoencoder | 87.8% |

In [ ]:
# BYOL koncepció - negatívok nélkül!
class BYOLConcept:
    """
    BYOL (Bootstrap Your Own Latent):
    - Online network: tanul
    - Target network: lassan követi az online-t (EMA)
    - Nincs szükség negatív mintákra!

    Miért nem omlik össze triviális megoldásra?
    → Az aszimmetrikus architektúra + EMA megakadályozza.
    """
    pass

# EMA (Exponential Moving Average) frissítés
def update_target_network(online_net, target_net, tau=0.99):
    """Target = tau * target + (1-tau) * online"""
    for online_param, target_param in zip(online_net.parameters(),
                                          target_net.parameters()):
        target_param.data = tau * target_param.data + (1 - tau) * online_param.data

print("BYOL architektúra:")
print("")
print("  Image ──┬──▶ [Online Encoder] ──▶ [Predictor] ──▶ z_online")
print("          │                                            │")
print("          │                                            ▼")
print("          └──▶ [Target Encoder] ──────────────────▶ z_target")
print("                    (EMA)")
print("")
print("Loss = MSE(z_online, z_target.detach())")

In [ ]:
# MAE (Masked Autoencoder) koncepció
def demonstrate_mae_masking(img, mask_ratio=0.75):
    """
    MAE: A kép 75%-át elrejtjük, és rekonstruáljuk.
    """
    img_array = np.array(img)
    h, w = img_array.shape[:2]
    patch_size = 16
    n_patches_h = h // patch_size
    n_patches_w = w // patch_size
    n_patches = n_patches_h * n_patches_w

    # Random maszk
    n_masked = int(n_patches * mask_ratio)
    mask_indices = np.random.choice(n_patches, n_masked, replace=False)

    masked_img = img_array.copy()
    for idx in mask_indices:
        i = idx // n_patches_w
        j = idx % n_patches_w
        masked_img[i*patch_size:(i+1)*patch_size,
                  j*patch_size:(j+1)*patch_size] = 128  # Szürke

    return Image.fromarray(masked_img), mask_ratio

# Vizualizáció
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(samples[1][1])
axes[0].set_title('Eredeti', fontsize=11)
axes[0].axis('off')

for i, ratio in enumerate([0.5, 0.75, 0.9]):
    masked, _ = demonstrate_mae_masking(samples[1][1], ratio)
    axes[i+1].imshow(masked)
    axes[i+1].set_title(f'{int(ratio*100)}% maszkolva', fontsize=11)
    axes[i+1].axis('off')

plt.suptitle('MAE: Masked Autoencoder - rekonstrukció maszkolt képből',
            fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Az encoder csak a látható patch-eket dolgozza fel → hatékony!")
print("A decoder rekonstruálja a teljes képet.")

## 5. Gyakorlati használat

### Pretrained SSL modellek

```python
# DINO v2 (Facebook Research)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

# CLIP (OpenAI) - multimodal
import clip
model, preprocess = clip.load("ViT-B/32")

# MAE (Facebook)
model = torch.hub.load('facebookresearch/mae', 'mae_vit_base_patch16')
```

### SSL → Downstream task pipeline

1. **Pretrain**: SSL nagy, címkézetlen adaton
2. **Linear probe**: Backbone fagyasztva, csak lineáris classifier tanítása
3. **Fine-tuning**: Teljes modell fine-tune kis, címkézett adaton

In [ ]:
# Linear probe vs Fine-tuning
class DownstreamClassifier(nn.Module):
    def __init__(self, pretrained_encoder, n_classes, freeze_backbone=True):
        super().__init__()
        self.encoder = pretrained_encoder

        # Backbone fagyasztása (linear probe)
        if freeze_backbone:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Classifier head
        self.classifier = nn.Linear(512, n_classes)  # backbone_dim → classes

    def forward(self, x):
        # Backbone reprezentáció (projection nélkül)
        features = self.encoder(x, return_projection=False)
        return self.classifier(features)

# Példa használat
pretrained = SimCLREncoder()  # SSL-lel pretrained

# Linear probe (gyors, kevés adat is elég)
linear_probe = DownstreamClassifier(pretrained, n_classes=10, freeze_backbone=True)

# Fine-tuning (lassabb, több adat kell)
finetuned = DownstreamClassifier(pretrained, n_classes=10, freeze_backbone=False)

print("Linear probe tanítható paraméterei:")
print(f"  {sum(p.numel() for p in linear_probe.parameters() if p.requires_grad):,}")

print("\nFine-tuning tanítható paraméterei:")
print(f"  {sum(p.numel() for p in finetuned.parameters() if p.requires_grad):,}")

## Összefoglalás

### SSL lényege
- Címkézetlen adatból tanulunk reprezentációkat
- A "címkéket" automatikusan generáljuk

### Főbb módszerek

| Kategória | Módszerek | Kulcs ötlet |
|-----------|-----------|-------------|
| Pretext tasks | Rotation, Jigsaw | Mesterséges feladatok |
| Contrastive | SimCLR, MoCo | Pozitív/negatív párok |
| Non-contrastive | BYOL, SimSiam | Csak pozitív párok |
| Masked modeling | MAE, BEiT | Rekonstrukció |

### Gyakorlati tanács

1. Használj **pretrained modelleket** (DINOv2, CLIP)
2. Kevés címkézett adatnál: **linear probe**
3. Több adatnál: **fine-tuning**
4. **Erős augmentáció** kulcsfontosságú!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
torch.manual_seed(42)

## 1. SSL motiváció

**Probléma**: Címkézett adatok drágák és limitáltak.

**Megoldás**: A modell saját maga generálja a "címkéket" a bemeneti adatból.

| Tanulási típus | Címke forrása | Példa |
|----------------|---------------|-------|
| Supervised | Emberi annotáció | ImageNet osztályok |
| Self-Supervised | Adat struktúrája | Kép augmentációk |

In [ ]:
# Teszt kép létrehozása
def create_sample_image():
    img = Image.new('RGB', (96, 96), (230, 230, 240))
    draw = ImageDraw.Draw(img)
    draw.ellipse([25, 25, 70, 70], fill=(200, 80, 80))
    draw.rectangle([50, 50, 80, 80], fill=(80, 150, 80))
    return img

sample = create_sample_image()
plt.figure(figsize=(3, 3))
plt.imshow(sample)
plt.title('Teszt kép')
plt.axis('off')
plt.show()

## 2. Pretext feladatok

Korai SSL megközelítések: mesterséges feladatok, amelyek hasznos reprezentációkat tanulnak.

In [ ]:
# Rotation prediction pretext task
def apply_rotation(img, angle_idx):
    """Forgatás: 0°, 90°, 180°, 270°"""
    angles = [0, 90, 180, 270]
    return img.rotate(angles[angle_idx]), angle_idx

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    rotated, label = apply_rotation(sample, i)
    ax.imshow(rotated)
    ax.set_title(f'Label: {i} ({i*90}°)')
    ax.axis('off')
plt.suptitle('Rotation Prediction - Pretext Task', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Jigsaw puzzle pretext task
def create_jigsaw(img, grid_size=3):
    """Kép feldarabolása és összekeverése."""
    img_array = np.array(img)
    h, w = img_array.shape[:2]
    ph, pw = h // grid_size, w // grid_size

    patches = []
    for i in range(grid_size):
        for j in range(grid_size):
            patch = img_array[i*ph:(i+1)*ph, j*pw:(j+1)*pw]
            patches.append(patch)

    # Összekeverés
    perm = np.random.permutation(len(patches))
    shuffled = [patches[i] for i in perm]

    # Visszarakás
    rows = []
    for i in range(grid_size):
        row = np.concatenate(shuffled[i*grid_size:(i+1)*grid_size], axis=1)
        rows.append(row)
    shuffled_img = np.concatenate(rows, axis=0)

    return Image.fromarray(shuffled_img), perm

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(sample)
axes[0].set_title('Eredeti')

for i in range(1, 3):
    jigsaw, perm = create_jigsaw(sample, 3)
    axes[i].imshow(jigsaw)
    axes[i].set_title(f'Kevert (perm: {list(perm)[:4]}...)')

for ax in axes:
    ax.axis('off')
plt.suptitle('Jigsaw Puzzle - Pretext Task', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Kontraszt tanulás (SimCLR)

**Alapötlet**: Ugyanannak a képnek két augmentált verzióját (pozitív pár) közel, különböző képekét (negatív párok) távol helyezzük el a reprezentációs térben.

$$\mathcal{L} = -\log \frac{\exp(sim(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(sim(z_i, z_k)/\tau)}$$

In [ ]:
# SimCLR augmentáció
simclr_transform = T.Compose([
    T.RandomResizedCrop(96, scale=(0.5, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.RandomGrayscale(p=0.2),
])

# Augmentált párok vizualizációja
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i in range(5):
    aug1 = simclr_transform(sample)
    aug2 = simclr_transform(sample)
    axes[0, i].imshow(aug1)
    axes[0, i].set_title(f'View 1.{i+1}')
    axes[1, i].imshow(aug2)
    axes[1, i].set_title(f'View 2.{i+1}')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('SimCLR: Pozitív párok (ugyanaz a kép, különböző augmentáció)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Egyszerű SimCLR-szerű encoder
class SimpleEncoder(nn.Module):
    def __init__(self, hidden_dim=128, proj_dim=64):
        super().__init__()
        # Backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1),  # 96->48
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1),  # 48->24
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1),  # 24->12
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        # Projection head
        self.projector = nn.Sequential(
            nn.Linear(128, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, proj_dim)
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)  # L2 normalizálás

def nt_xent_loss(z1, z2, temperature=0.5):
    """NT-Xent (InfoNCE) loss."""
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.t()) / temperature

    # Maszk: diagonális és pozitív párok kizárása a nevezőből
    mask = torch.eye(2*batch_size, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)

    # Pozitív párok
    pos_mask = torch.zeros(2*batch_size, 2*batch_size, dtype=torch.bool)
    for i in range(batch_size):
        pos_mask[i, batch_size + i] = True
        pos_mask[batch_size + i, i] = True

    pos_sim = sim[pos_mask].view(2*batch_size, 1)
    neg_sim = sim[~mask & ~pos_mask].view(2*batch_size, -1)

    logits = torch.cat([pos_sim, neg_sim], dim=1)
    labels = torch.zeros(2*batch_size, dtype=torch.long)

    return F.cross_entropy(logits, labels)

# Teszt
encoder = SimpleEncoder()
x = torch.randn(4, 3, 96, 96)
z = encoder(x)
print(f"Projection output shape: {z.shape}")

## 4. Modern SSL módszerek

| Módszer | Év | Kulcs újítás |
|---------|-----|-------------|
| SimCLR | 2020 | Erős augmentáció + nagy batch |
| MoCo | 2020 | Momentum encoder, queue |
| BYOL | 2020 | Nincs negatív minta! |
| SwAV | 2020 | Online klaszterezés |
| DINO | 2021 | Self-distillation, ViT |
| MAE | 2022 | Masked autoencoder |

In [ ]:
def visualize_ssl_landscape():
    fig, ax = plt.subplots(figsize=(14, 6))

    methods = [
        ('Rotation\nPrediction', 0.08, 0.3, 'lightcoral', '2018'),
        ('SimCLR', 0.22, 0.7, 'lightblue', '2020'),
        ('MoCo v2', 0.36, 0.65, 'lightblue', '2020'),
        ('BYOL', 0.50, 0.75, 'lightgreen', '2020'),
        ('DINO', 0.64, 0.8, 'plum', '2021'),
        ('MAE', 0.78, 0.85, 'lightyellow', '2022'),
        ('DINOv2', 0.92, 0.9, 'plum', '2023'),
    ]

    for name, x, y, color, year in methods:
        ax.add_patch(plt.Circle((x, y), 0.06, facecolor=color, edgecolor='black'))
        ax.text(x, y, name, ha='center', va='center', fontsize=8, fontweight='bold')
        ax.text(x, y-0.12, year, ha='center', fontsize=8)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Idő →', fontsize=12)
    ax.set_ylabel('ImageNet Linear Probe Acc →', fontsize=12)
    ax.set_title('Self-Supervised Learning fejlődése', fontsize=14, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()

visualize_ssl_landscape()

## Összefoglalás

**SSL előnyök:**
- Címkézetlen adatokból tanul
- Jobb általánosítás
- Transfer learning-re kiváló

**Kulcs technikák:**
- Erős augmentáció
- Kontraszt tanulás (pozitív/negatív párok)
- Momentum encoder
- Self-distillation

**Gyakorlati használat:**
```python
# DINO pretrained model (torchvision)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
```